# Imports  + seed

In [2]:
!pip install yfinance ta --quiet
print("Done installing.")

  Preparing metadata (setup.py) ... done
Done installing.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import yfinance as yf
from tensorflow.keras import layers
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands, AverageTrueRange

#understood it wa relevant to make results repeatable so :
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


# Question 1

## Preparing the data  

My First Approah here is to to get the data from the package yahoo finance.
The stock I chose is 'HD' (Home Depot) as my last name is Hamra

In [4]:
# I use the yahou finance package

Stock = "HD"
start_date  = "2010-01-01"
end_date    = "2026-05-31"

# we put our financial data into a pandas data frame
df = yf.download(Stock, start=start_date, end=end_date, auto_adjust=True)
print(df.shape)
df.head(-1)


[*********************100%***********************]  1 of 1 completed

(4126, 5)


Price,Close,High,Low,Open,Volume
Ticker,HD,HD,HD,HD,HD
Date,,,,,
2010-01-04,19.297073,19.620149,19.216304,19.620149,13120900
2010-01-05,19.438419,19.512458,19.041306,19.337458,15594300
2010-01-06,19.371115,19.519191,19.317269,19.438421,8833200
2010-01-07,19.599960,19.667267,19.337460,19.424961,12058200
2010-01-08,19.505730,19.680729,19.310538,19.519191,15609800
...,...,...,...,...,...
2026-05-21,311.443970,311.652398,302.113957,304.079197,4516600
2026-05-22,310.739258,312.615171,309.141228,312.158604,2963800


In [5]:
# we see above that there is a double column at the top named 'Ticker' which ill remove for better practise later:
df.columns = df.columns.get_level_values(0)
print(df.shape)
df.head(-1)


(4126, 5)


Price,Close,High,Low,Open,Volume
Date,,,,,
2010-01-04,19.297073,19.620149,19.216304,19.620149,13120900
2010-01-05,19.438419,19.512458,19.041306,19.337458,15594300
2010-01-06,19.371115,19.519191,19.317269,19.438421,8833200
2010-01-07,19.599960,19.667267,19.337460,19.424961,12058200
2010-01-08,19.505730,19.680729,19.310538,19.519191,15609800
...,...,...,...,...,...
2026-05-21,311.443970,311.652398,302.113957,304.079197,4516600
2026-05-22,310.739258,312.615171,309.141228,312.158604,2963800
2026-05-26,308.228088,314.431558,306.659834,312.714427,6978200


In [6]:
# reorder them :
df = df[["Open", "High", "Low", "Close", "Volume"]]
df.head()

Price,Open,High,Low,Close,Volume
Date,,,,,
2010-01-04,19.620149,19.620149,19.216304,19.297073,13120900
2010-01-05,19.337458,19.512458,19.041306,19.438419,15594300
2010-01-06,19.438421,19.519191,19.317269,19.371115,8833200
2010-01-07,19.424961,19.667267,19.337460,19.599960,12058200
2010-01-08,19.519191,19.680729,19.310538,19.505730,15609800


### Getting new indicators columns

In [8]:
# Now im going to add columns to the datasets that have been asked from question 1 which are :
#MOVING AVERAGES, RSI, MACD, Volatility 1 (bollinger band with) and volatility 2 (Average true Range)

# Most indicators use the closing price, so i put it in a variable
closing_price = indicators["Close"]

# -- Moving Averages --
# Average of the last N days. Bigger N = smoother / slower-reacting, fro now now i added 2 10 and 30 maybe it will get adjusted
indicators["SMA_10"] = closing_price.rolling(window=10).mean()   # short-term trend
indicators["SMA_20"] = closing_price.rolling(window=20).mean()   # longer-term trend

# EMA = moving average that weights recent days more (reacts faster than SMA)
indicators["EMA_10"] = closing_price.ewm(span=10, adjust=False).mean()

# -- RSI (momentum, 0 to 100) --
daily_change = closing_price.diff()

# Split each day's change into "gain" (up days) and "loss" (down days)
gains  = daily_change.clip(lower=0)          # keep ups, set downs to 0
losses = -daily_change.clip(upper=0)         # keep downs (flipped positive), ups to 0

# Smooth the gains and losses over 14 days (the standard RSI smoothing)
average_gain = gains.ewm(alpha=1/14, adjust=False).mean()
average_loss = losses.ewm(alpha=1/14, adjust=False).mean()

relative_strength = average_gain / average_loss
indicators["RSI_14"] = 100 - (100 / (1 + relative_strength))

# -- MACD (trend) --
# Compare a fast EMA (12 days) to a slow EMA (26 days)
fast_ema = closing_price.ewm(span=12, adjust=False).mean()
slow_ema = closing_price.ewm(span=26, adjust=False).mean()

indicators["MACD"]        = fast_ema - slow_ema                              # the MACD line
indicators["MACD_signal"] = indicators["MACD"].ewm(span=9, adjust=False).mean()  # smoothed MACD
indicators["MACD_diff"]   = indicators["MACD"] - indicators["MACD_signal"]   # gap between them

# --VOLATILITY 1: Bollinger Band width --
average_20  = closing_price.rolling(window=20).mean()   # middle line
spread_20   = closing_price.rolling(window=20).std()    # how spread out prices are
upper_band  = average_20 + 2 * spread_20
lower_band  = average_20 - 2 * spread_20
indicators["BB_width"] = (upper_band - lower_band) / average_20   # wide = volatile

# -- VOLATILITY 2: Average True Range (ATR) --
high_price      = indicators["High"]
low_price       = indicators["Low"]
yesterday_close = closing_price.shift(1)   # shift down by 1 = yesterday's value

# True Range = the biggest of these three daily distances
true_range = pd.concat([
    high_price - low_price,                  # today's high-to-low
    (high_price - yesterday_close).abs(),    # gap up from yesterday
    (low_price  - yesterday_close).abs(),    # gap down from yesterday
], axis=1).max(axis=1)

indicators["ATR_14"] = true_range.ewm(alpha=1/14, adjust=False).mean()

# -- remove warm-up rows --
# The first ~26 rows are blank because long indicators need history to start.
indicators = indicators.dropna().reset_index()

print(indicators.shape)
indicators.head()


NameError: name 'indicators' is not defined